In [0]:
def run_pipeline(spark, config, auth, headers, base, version, endpoint, target_db, log_table, results=None):

    if results is None:
        results = {}

    # Step 1 — build URL
    v = agile_version if config.get("version_key") == "agile_version" else version
    url = build_url(base, v, endpoint[config["endpoint_key"]])
    all_data = []

    # Step 2 — fetch based on strategy
    if config["strategy"] == "per_parent":
        parent_cfg     = config["parent"]
        parent_source  = parent_cfg["source"]
        parent_field   = parent_cfg["field"]
        param_name     = parent_cfg["param_name"]
        param_template = parent_cfg["param_template"]

        parent_df     = results[parent_source]
        parent_values = [row[parent_field] for row in parent_df.collect()]

        for value in parent_values:
            params = {
                param_name: param_template.replace("{VALUE}", value),
                "fields"  : config.get("fields", "")
            }
            batch = fetch_paginated_data(url, auth, headers, params, config["data_key"])
            all_data.extend(apply_parser(batch, config["parser"]))

    elif config["strategy"] == "flat":
        items = fetch_paginated_data(url, auth, headers, {}, config["data_key"])
        all_data.extend(apply_parser(items, config["parser"]))

    # Step 3 — create raw DataFrame + audit columns
    df_raw = create_df(spark, all_data, schema=config.get("df_schema"))
    df_raw = (
        df_raw
        .withColumn("ingestion_timestamp", F.lit(datetime.now().isoformat()))
        .withColumn("pipeline_name"      , F.lit(config["name"]))
        .withColumn("source"             , F.lit("jira"))
    )

    # Step 4 — write raw table if exists
    if config["raw_table"]:
        write_table(
            spark, df_raw,
            f"{target_db}.{config['raw_table']}",
            mode=config.get("raw_write_mode", "append")
        )

    # Step 5 — store in results for child pipelines
    if config.get("result_fields"):
        df_for_results = df_raw
        for field in config["result_fields"]:
            df_for_results = df_for_results.withColumn(
                field,
                F.get_json_object(F.col("raw_json"), f"$.{field}")
            )
        results[config["name"]] = df_for_results
    else:
        results[config["name"]] = df_raw

    # Step 6 — if no selected_table, log and return early
    if not config["selected_table"]:
        log_pipeline_run(
            spark,
            f"{target_db}.{log_table}",
            pipeline_name     = config["name"],
            status            = "SUCCESS",
            records_processed = len(all_data),
            mode              = "append"
        

        )
        return df_raw

    # Step 7 — parse JSON into schema
    df_parsed = df_raw.withColumn("parsed", F.from_json("raw_json", config["schema"]))

    # Step 8 — dynamic column selection
    dynamic_cols = [F.col(path).alias(alias) for path, alias in config["selected_columns"]]
    audit_cols   = [
        F.col("ingestion_timestamp"),
        F.col("pipeline_name"),
        F.col("source")
    ]
    df_selected = df_parsed.select(dynamic_cols + audit_cols)

    # Step 9 — write selected table
    write_table(
        spark, df_selected,
        f"{target_db}.{config['selected_table']}",
        mode      = config.get("selected_write_mode", "append"),
        merge_key = config.get("merge_key")
    )

    # Step 10 — log run
    log_pipeline_run(
        spark,
        f"{target_db}.{log_table}",
        pipeline_name     = config["name"],
        status            = "SUCCESS",
        records_processed = len(all_data),
        mode              = "append" 
    )

    return df_selected